# LNCLIP-DF v2 — Dual-Input Deepfake Detection

**Architecture:** CLIP ViT-L/14 + LN-tuning + Dual-Input (Face + Full Frame) + Angular Margin Warmup

**Optimizations:** Local SSD cache, batch=8, early stopping (patience=5)

**Target runtime:** < 5 hours total on T4
- Cell 1-3 (setup): ~5 min
- Cell 4 (preprocessing, first run only): ~2.5h | subsequent runs: 0 min (cached)
- Cell 5 (copy to local SSD): ~10-15 min
- Cell 6 (build model): ~2 min
- Cell 7 (training, 20 epochs max): ~1.5-2h from SSD
- Cell 8-9 (eval + results): ~10 min

**Total first run:** ~4.5h | **Re-runs (cache exists):** ~2h

In [ ]:
# ═══ CELL 1: Setup ═══════════════════════════════════════════
import torch
assert torch.cuda.is_available(), 'Need T4 GPU → Runtime → Change runtime type → T4'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')
!pip install -q open_clip_torch insightface onnxruntime-gpu opencv-python-headless scipy scikit-learn tqdm pandas matplotlib
print('✓ Dependencies installed')

In [ ]:
# ═══ CELL 2: Mount + Clone ═══════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, random
from pathlib import Path

if not os.path.exists('/content/eidon-deepfake'):
    !git clone https://github.com/rxbinsingh/eidon-deepfake /content/eidon-deepfake
else:
    !git -C /content/eidon-deepfake pull --quiet

os.chdir('/content/eidon-deepfake')
sys.path.insert(0, '/content/eidon-deepfake')

BASE      = '/content/drive/MyDrive/VIPER'
DATA_DIRS = [f'{BASE}/dataset_production', f'{BASE}/dataset_week2', f'{BASE}/dfd_dataset']
CACHE_DIR = f'{BASE}/cache_dual'       # Drive cache (persistent)
CACHE_LOCAL = '/content/cache_local'   # Local SSD cache (fast I/O)
SAVE_DIR  = f'{BASE}/checkpoints_v2'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CACHE_LOCAL, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

import pandas as pd
total = 0
for d in DATA_DIRS:
    if os.path.exists(f'{d}/metadata.csv'):
        n = len(pd.read_csv(f'{d}/metadata.csv'))
        total += n
        print(f'  ✓ {Path(d).name}: {n} videos')
    else:
        print(f'  ✗ {Path(d).name}: NOT FOUND')
print(f'Total: {total}')

In [ ]:
# ═══ CELL 3: Find all videos ════════════════════════════════
import numpy as np

def get_all_videos():
    samples = []
    for d in DATA_DIRS:
        meta_path = f'{d}/metadata.csv'
        if not os.path.exists(meta_path): continue
        meta = pd.read_csv(meta_path)
        found, missed = 0, 0
        for _, row in meta.iterrows():
            label_str = row.get('label', 'unknown')
            label = 0 if label_str == 'real' else 1
            cat = row.get('category', label_str)
            fname = row['filename']
            possible = [f'{d}/{cat}/{fname}', f'{d}/{label_str}/{fname}']
            if 'original_path' in row and pd.notna(row['original_path']):
                possible.append(f'{d}/{row["original_path"]}')
                possible.append(f'{d}/{Path(row["original_path"]).parent}/{fname}')
            if 'dfd' in d.lower() or 'DFD' in d:
                possible += [f'{d}/DFD_original sequences/{fname}',
                             f'{d}/DFD_manipulated_sequences/{fname}',
                             f'{d}/DFD_manipulated_sequences/DFD_manipulated_sequences/{fname}']
            for vp in possible:
                if os.path.exists(vp):
                    samples.append((vp, label, cat)); found += 1; break
            else: missed += 1
        print(f'  {Path(d).name}: found={found}, missed={missed}')
    return samples

all_videos = get_all_videos()
n_real = sum(1 for _,l,_ in all_videos if l==0)
n_fake = sum(1 for _,l,_ in all_videos if l==1)
print(f'\nTotal: {len(all_videos)} | Real: {n_real} | Fake: {n_fake}')

In [ ]:
# ═══ CELL 4: Preprocess to Drive cache (skip if already done)
# ~2.5h first run. Skips cached videos automatically.
import cv2
from tqdm import tqdm
from src.preprocessing import preprocess_video
try:
    import src.anchor_extractor as ae
    ae.build_biomech_anchor = lambda frames: None
except: pass

already = len(glob.glob(f'{CACHE_DIR}/*.npz'))
to_process = len(all_videos) - already
print(f'Cached: {already} | To process: {to_process}')
if to_process == 0:
    print('All cached — skip to Cell 5')
else:
    ok, fail = 0, 0
    for vp, label, cat in tqdm(all_videos, desc='Preprocessing'):
        cp = f'{CACHE_DIR}/{Path(vp).stem}.npz'
        if os.path.exists(cp): ok += 1; continue
        try:
            p = preprocess_video(vp, num_frames=16, n_anchor=8)
            if not p['valid']: fail += 1; continue
            full_frames = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['full_frames']]
            save_data = {'full_frames': np.stack(full_frames), 'label': np.array(label),
                         'face_valid': np.array(p['face_valid'])}
            if p['face_valid'] and p['face_crops']:
                save_data['face_crops'] = np.stack([cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['face_crops']])
            np.savez_compressed(cp, **save_data)
            ok += 1
        except: fail += 1
    print(f'Done. OK={ok} Fail={fail}')

In [ ]:
# ═══ CELL 5: Copy cache to local SSD (CRITICAL for speed)
# Drive I/O is slow (7s/batch). Local SSD is 5x faster (~1.5s/batch).
# ~10-15 min copy. Training will be 3x faster after this.

cached_files = glob.glob(f'{CACHE_DIR}/*.npz')
local_files = glob.glob(f'{CACHE_LOCAL}/*.npz')
print(f'Drive cache: {len(cached_files)} files')
print(f'Local cache: {len(local_files)} files')

if len(local_files) < len(cached_files):
    print('Copying to local SSD...')
    !cp {CACHE_DIR}/*.npz {CACHE_LOCAL}/
    print(f'Done. Local: {len(glob.glob(f"{CACHE_LOCAL}/*.npz"))} files')
else:
    print('Local cache up to date.')

In [ ]:
# ═══ CELL 6: Build Model + Datasets ═════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from PIL import Image
import open_clip
from src.lnclip_model import LNCLIPDeepfakeDetector, ArcMarginProduct, SphericalAugmentation

device = 'cuda'
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ── Compression Augmentation ──────────────────────────────────
class CompressAug:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        arr = np.array(img)
        if random.random() < 0.5:
            q = random.randint(30, 95)
            _, enc = cv2.imencode('.jpg', cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
            arr = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if random.random() < 0.3:
            s = random.uniform(0.5, 2.0); k = int(s*4)|1
            arr = cv2.GaussianBlur(arr, (k,k), s)
        if random.random() < 0.3:
            sc = random.uniform(0.6, 0.9); h,w = arr.shape[:2]
            arr = cv2.resize(cv2.resize(arr, (int(w*sc),int(h*sc))), (w,h))
        return Image.fromarray(arr)

CLIP_TF = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.48145466,0.4578275,0.40821073],[0.26862954,0.26130258,0.27577711])])

# ── Dual Dataset (reads from local SSD) ────────────────────────
class DualDataset(Dataset):
    def __init__(self, samples, cache_dir, train=True):
        self.samples = [(p,l) for p,l,_ in samples
                        if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir = cache_dir
        self.aug = CompressAug(0.5) if train else None
        self.flip = T.RandomHorizontalFlip(0.5) if train else T.RandomHorizontalFlip(0)
        print(f'  {"Train" if train else "Eval"}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        vp, label = self.samples[idx]
        data = np.load(f'{self.cache_dir}/{Path(vp).stem}.npz')
        full_frames = data['full_frames']
        face_valid = bool(data['face_valid'])
        face_crops = data.get('face_crops', None)

        def proc(arr, n=16):
            T_ = min(len(arr), n)
            ts = []
            for i in range(T_):
                img = Image.fromarray(arr[i])
                if self.aug: img = self.aug(img)
                img = self.flip(img)
                ts.append(CLIP_TF(img))
            while len(ts) < n: ts.append(ts[-1])
            return torch.stack(ts[:n])

        full_t = proc(full_frames)
        face_t = proc(face_crops) if (face_valid and face_crops is not None and len(face_crops)>=4) else torch.zeros_like(full_t)
        return {'face': face_t, 'full': full_t, 'label': torch.tensor(label, dtype=torch.long)}

# ── Split ──────────────────────────────────────────────────────
shuffled = all_videos.copy(); random.shuffle(shuffled)
n = len(shuffled)
train_s = shuffled[:int(.7*n)]; val_s = shuffled[int(.7*n):int(.8*n)]; test_s = shuffled[int(.8*n):]
print(f'Split: Train={len(train_s)} Val={len(val_s)} Test={len(test_s)}')

# Use LOCAL SSD for fast I/O
train_ds = DualDataset(train_s, CACHE_LOCAL, train=True)
val_ds   = DualDataset(val_s,   CACHE_LOCAL, train=False)
test_ds  = DualDataset(test_s,  CACHE_LOCAL, train=False)

# batch=8 for speed (2x face+full per sample but VRAM allows it)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

# ── Build Model ────────────────────────────────────────────────
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
clip_model = clip_model.to(device).eval()

model = LNCLIPDeepfakeDetector(clip_model, num_trainable_layers=6).to(device)

# Replace angular margin with simple linear head for stable training
model.classifier = nn.Linear(model.combined_dim, 2).to(device)
nn.init.xavier_uniform_(model.classifier.weight)
nn.init.zeros_(model.classifier.bias)
model.sphere_aug.noise_std = 0.0  # disable spherical noise initially

# Override forward for simple linear mode
def simple_forward(face, full, labels=None):
    emb = model.encode_frames(face, full)
    return model.classifier(emb), emb
model.forward = simple_forward

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,} params ({trainable/sum(p.numel() for p in model.parameters())*100:.3f}%)')

In [ ]:
# ═══ CELL 7: Training with Hyperparameter Tuning ═════════════
# Strategy:
#   Phase 1 (epochs 1-5):  Simple CE loss, LR=3e-4, LN-tuning enabled
#   Phase 2 (epochs 6-20): Add spherical noise, LR decays via cosine
#   Early stopping: patience=5 (stops if no Val AUC improvement for 5 epochs)
#   Total time: ~1.5h from local SSD

import time, json

EPOCHS  = 20
PATIENCE = 5

# Class-weighted loss (real=362, fake=694 → weight real higher)
n_real_tr = sum(1 for _,l,_ in train_s if l==0)
n_fake_tr = sum(1 for _,l,_ in train_s if l==1)
class_weights = torch.tensor([n_fake_tr/n_real_tr, 1.0]).to(device)  # upweight real class
print(f'Class weights: real={class_weights[0]:.2f}, fake={class_weights[1]:.2f}')

# Discriminative LR: LN params slower, classifier faster
ln_params = [p for n_, p in model.named_parameters()
             if p.requires_grad and 'classifier' not in n_]
head_params = list(model.classifier.parameters())
optimizer = AdamW([
    {'params': ln_params,   'lr': 1e-4},
    {'params': head_params, 'lr': 3e-4},
], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

def train_epoch(epoch):
    model.train()
    # Phase 2: enable spherical noise after epoch 5
    if epoch > 5:
        model.sphere_aug.noise_std = 0.03
    all_l, all_p, loss_sum = [], [], 0.0
    for batch in tqdm(train_loader, leave=False):
        face = batch['face'].to(device)
        full = batch['full'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, emb = model(face, full, labels)
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()
        all_l.extend(labels.cpu().numpy())
        all_p.extend(F.softmax(logits.detach(), dim=-1)[:,1].cpu().numpy())
    auc = roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0
    return loss_sum/len(train_loader), auc

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    all_l, all_p = [], []
    for batch in loader:
        face = batch['face'].to(device)
        full = batch['full'].to(device)
        with torch.amp.autocast('cuda'):
            logits, _ = model(face, full, labels=None)
        all_l.extend(batch['label'].numpy())
        all_p.extend(F.softmax(logits, dim=-1)[:,1].cpu().numpy())
    auc = roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0
    preds = [1 if p>.5 else 0 for p in all_p]
    return auc, accuracy_score(all_l, preds), all_l, all_p

best_auc, patience_ct = 0.0, 0
history = []
t_start = time.time()
print(f'Training (max {EPOCHS} epochs, patience={PATIENCE})...\n')

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_auc = train_epoch(epoch)
    vl_auc, vl_acc, _, _ = eval_epoch(val_loader)
    scheduler.step()
    elapsed = time.time() - t0
    total_elapsed = (time.time() - t_start)/60
    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc; patience_ct = 0
        torch.save(model.state_dict(), f'{SAVE_DIR}/best.pt')
        flag = '  ← best'
    else: patience_ct += 1
    phase = 'CE+noise' if epoch > 5 else 'CE'
    print(f'Ep {epoch:02d}/{EPOCHS} [{phase}] Loss={tr_loss:.4f} '
          f'Train={tr_auc:.4f} Val={vl_auc:.4f} Acc={vl_acc:.3f} '
          f'({elapsed:.0f}s | {total_elapsed:.0f}min total){flag}')
    history.append({'epoch':epoch,'tr_loss':tr_loss,'tr_auc':tr_auc,'vl_auc':vl_auc,'vl_acc':vl_acc})
    if patience_ct >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

torch.save(model.state_dict(), f'{SAVE_DIR}/final.pt')
with open(f'{SAVE_DIR}/history.json','w') as f: json.dump(history, f, indent=2)
print(f'\nBest Val AUC: {best_auc:.4f} | Total time: {(time.time()-t_start)/60:.0f} min')

In [ ]:
# ═══ CELL 8: Final Test Evaluation + TTA ═════════════════════
model.load_state_dict(torch.load(f'{SAVE_DIR}/best.pt', map_location=device))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test + TTA'):
        face = batch['face'].to(device)
        full = batch['full'].to(device)
        with torch.amp.autocast('cuda'):
            l1, _ = model(face, full, labels=None)
            l2, _ = model(torch.flip(face,[-1]), torch.flip(full,[-1]), labels=None)
        probs = (F.softmax(l1,dim=-1)[:,1] + F.softmax(l2,dim=-1)[:,1]) / 2
        all_labels.extend(batch['label'].numpy())
        all_probs.extend(probs.cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
preds = [1 if p>.5 else 0 for p in all_probs]
acc = accuracy_score(all_labels, preds)
cm = confusion_matrix(all_labels, preds)

print(f'\n{"═"*52}')
print(f'  LNCLIP-DF v2 — FINAL TEST RESULTS')
print(f'{"═"*52}')
print(f'  AUC-ROC:  {auc:.4f}')
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'  TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}')
print(f'  FPR: {cm[0,1]/(cm[0,0]+cm[0,1])*100:.1f}% | FNR: {cm[1,0]/(cm[1,0]+cm[1,1])*100:.1f}%')
print(f'\n{classification_report(all_labels, preds, target_names=["Real","Fake"])}')

In [ ]:
# ═══ CELL 9: Per-Category + Training Curves + Save ═══════════
import matplotlib.pyplot as plt

# Per-category breakdown
cat_l, cat_p = {}, {}
for (vp,label,cat), prob in zip(test_s, all_probs[:len(test_s)]):
    if cat not in cat_l: cat_l[cat], cat_p[cat] = [], []
    cat_l[cat].append(label); cat_p[cat].append(prob)

print(f'{"Category":<28} {"AUC":>8} {"N":>5}')
print('─'*45)
real_l, real_p = cat_l.get('real',[]), cat_p.get('real',[])
per_type = {}
for cat in sorted(cat_l.keys()):
    if cat == 'real': continue
    cl = real_l + cat_l[cat]; cp_ = real_p + cat_p[cat]
    if len(set(cl)) < 2: continue
    a = roc_auc_score(cl, cp_)
    per_type[cat] = round(a, 4)
    print(f'{cat:<28} {a:>8.4f} {len(cat_l[cat]):>5}')
print('─'*45)
print(f'{"ALL":<28} {auc:>8.4f} {len(all_labels):>5}')

# Training curves
epochs_done = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_done, [h['tr_loss'] for h in history], 'b-o', label='Train')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_done, [h['tr_auc'] for h in history], 'b-o', label='Train')
axes[1].plot(epochs_done, [h['vl_auc'] for h in history], 'r-o', label='Val')
axes[1].axhline(best_auc, color='green', linestyle='--', label=f'Best={best_auc:.4f}')
axes[1].set_title('AUC'); axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0.5, 1.0)
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle(f'LNCLIP-DF v2 | Test AUC: {auc:.4f} | Acc: {acc*100:.1f}%', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Save final results
results = {
    'model': 'LNCLIP-DF v2 (Dual-Input + LN-tuning + Class Weighting + Phase Training)',
    'baseline_auc': 0.9535,
    'test_auc': round(auc, 4),
    'improvement': round(auc - 0.9535, 4),
    'accuracy': round(acc, 4),
    'best_val_auc': round(best_auc, 4),
    'per_type_auc': per_type,
    'confusion_matrix': cm.tolist(),
    'hyperparameters': {
        'ln_lr': '1e-4', 'head_lr': '3e-4', 'weight_decay': '1e-4',
        'class_weight_real': round(class_weights[0].item(), 3),
        'batch_size': 8, 'early_stopping_patience': PATIENCE,
        'phase_switch_epoch': 5, 'sphere_noise_phase2': 0.03,
    },
}
with open(f'{SAVE_DIR}/results_v2.json','w') as f: json.dump(results, f, indent=2)
print(f'\n✓ Results saved to {SAVE_DIR}/')
print(f'\nBaseline AUC: 0.9535 → New AUC: {auc:.4f} (Δ {auc-0.9535:+.4f})')